# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

##Method: Random Forest Classifier

I chose a Random Forest classifier because the modeling question is whether a content item is likely to be declining based on its available content and performance signals. The relationship between these signals may not be purely linear, so a tree-based model can capture nonlinear interactions between variables such as content age, CTR, impressions, engagement, and position.

I will use the model to rank content by its probability of decline. This fits the same decision-support use case as my Week-4 baseline, which ranks content for refresh review.

I will compare the Random Forest ranking with the Week-4 rule-based baseline using the same test data and Precision@50 as the main ranking metric.

Important leakage rule: trend_direction and trend_pct are excluded from the model features because the target is derived from trend_direction. The official starter pipeline follows the same rule.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [16]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

from sklearn.model_selection import train_test_split

# Create the binary target.
# 1 = declining, 0 = not declining
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

print("Target distribution:")
print(df["is_declining"].value_counts())
print("\nTarget percentage:")
print(df["is_declining"].value_counts(normalize=True))

Target distribution:
is_declining
1    16262
0    13738
Name: count, dtype: int64

Target percentage:
is_declining
1    0.542067
0    0.457933
Name: proportion, dtype: float64


##Split strategy: client holdout

I will use a client-level holdout rather than randomly splitting individual rows. This prevents pages from the same client appearing in both training and testing data, which would make the evaluation less independent.

Approximately 20% of clients will be held out for testing. The remaining clients will be used for training. This follows the same evaluation idea used by the starter pipeline.

In [17]:
# Client-aware 80/20 split

rng = np.random.default_rng(42)

clients = df["client_id"].fillna("unknown").astype(str).unique()
clients = rng.permutation(clients)

test_client_count = max(1, int(round(len(clients) * 0.20)))

test_clients = set(clients[:test_client_count])

test_mask = df["client_id"].fillna("unknown").astype(str).isin(test_clients)

train_df = df.loc[~test_mask].copy()
test_df = df.loc[test_mask].copy()

print("Split strategy: client holdout")
print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))
print("Training clients:", train_df["client_id"].nunique())
print("Testing clients:", test_df["client_id"].nunique())

print("\nTarget distribution - training:")
print(train_df["is_declining"].value_counts(normalize=True))

print("\nTarget distribution - testing:")
print(test_df["is_declining"].value_counts(normalize=True))

Split strategy: client holdout
Training rows: 27675
Testing rows: 2325
Training clients: 26
Testing clients: 6

Target distribution - training:
is_declining
1    0.554761
0    0.445239
Name: proportion, dtype: float64

Target distribution - testing:
is_declining
0    0.609032
1    0.390968
Name: proportion, dtype: float64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# Features available before the decline outcome is known.
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

feature_columns = numeric_features + categorical_features

X_train = train_df[feature_columns]
y_train = train_df["is_declining"]

X_test = test_df[feature_columns]
y_test = test_df["is_declining"]

print("Number of features before encoding:", len(feature_columns))
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Number of features before encoding: 26
Training shape: (27675, 26)
Testing shape: (2325, 26)


In [19]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_features
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ))
            ]),
            categorical_features
        )
    ]
)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=25,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

pipeline.fit(X_train, y_train)

print("Random Forest training completed.")

Random Forest training completed.


In [20]:
test_probabilities = pipeline.predict_proba(X_test)[:, 1]

test_results = test_df[
    ["content_id", "client_id", "is_declining"]
].copy()

test_results["model_score"] = test_probabilities

test_results = test_results.sort_values(
    "model_score",
    ascending=False
)

test_results.head(20)

,content_id,client_id,is_declining,model_score
11184,content_52b1c884e871,client_f74efabef1,1,0.788612
1085,content_0cf67ec37ab8,client_f74efabef1,1,0.783715
1476,content_6e792cf3ce56,client_f74efabef1,1,0.783245
20830,content_575fd096bff5,client_f74efabef1,1,0.782524
1958,content_6e17dbac0491,client_f74efabef1,1,0.780211
25913,content_331182ca4cae,client_f74efabef1,0,0.774789
28951,content_83be0494c955,client_f74efabef1,1,0.758843
88,content_998f6f88784c,client_f74efabef1,1,0.746777
21530,content_b15a8dbdf66f,client_f74efabef1,0,0.744770
680,content_efce5533859f,client_f74efabef1,1,0.743296


In [21]:
comparison_df = test_df[
    [
        "content_id",
        "client_id",
        "is_declining",
        "days_since_last_update",
        "content_age_days",
        "ctr",
        "impressions_90d"
    ]
].copy()

comparison_df["model_score"] = test_probabilities

comparison_df["baseline_score"] = (
    (comparison_df["days_since_last_update"] * 0.35)
    + (comparison_df["content_age_days"] * 0.20)
    + ((100 - comparison_df["ctr"]) * 0.25)
    + (np.log1p(comparison_df["impressions_90d"]) * 5)
)

comparison_df.head()

,content_id,client_id,is_declining,days_since_last_update,content_age_days,ctr,impressions_90d,model_score,baseline_score
11,content_5a3e876cf7f7,client_d4735e3a26,0,20,312,0.00,1,0.044391,97.865736
48,content_326fa2fa449f,client_98a3ab7c34,1,1,91,0.00,4,0.345963,51.597190
61,content_d99c66ea5462,client_d4735e3a26,0,20,130,11.11,9,0.351378,66.735425
63,content_d5d3c2e98937,client_f74efabef1,1,20,175,0.00,63,0.654886,87.794415
69,content_95d488a56079,client_f74efabef1,0,8,140,0.89,564,0.606492,87.261629


In [22]:
def precision_at_k(data, score_column, k=50):
    ranked = data.sort_values(score_column, ascending=False).head(k)
    return ranked["is_declining"].mean()

model_p20 = precision_at_k(comparison_df, "model_score", 20)
model_p50 = precision_at_k(comparison_df, "model_score", 50)
model_p100 = precision_at_k(comparison_df, "model_score", 100)

baseline_p20 = precision_at_k(comparison_df, "baseline_score", 20)
baseline_p50 = precision_at_k(comparison_df, "baseline_score", 50)
baseline_p100 = precision_at_k(comparison_df, "baseline_score", 100)

results_table = pd.DataFrame({
    "Method": [
        "Week-4 Baseline",
        "Random Forest"
    ],
    "Precision@20": [
        baseline_p20,
        model_p20
    ],
    "Precision@50": [
        baseline_p50,
        model_p50
    ],
    "Precision@100": [
        baseline_p100,
        model_p100
    ]
})

results_table

,Method,Precision@20,Precision@50,Precision@100
0,Week-4 Baseline,0.40,0.30,0.28
1,Random Forest,0.75,0.76,0.74


##Model vs Baseline

The Random Forest outperformed the Week-4 baseline at all three ranking cutoffs.

Precision@20 increased from 0.40 to 0.75, Precision@50 increased from 0.30 to 0.76, and Precision@100 increased from 0.28 to 0.74.

This indicates that, on this particular client-held-out test set, the Random Forest was more effective at placing declining content near the top of the review queue than the rule-based baseline.

These results should be treated as measured results from this test split rather than evidence that the model will perform identically on future data.

In [23]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

test_predictions = (test_probabilities >= 0.5).astype(int)

print("Random Forest Metrics")
print("---------------------")
print("Accuracy:", round(accuracy_score(y_test, test_predictions), 4))
print("Precision:", round(precision_score(y_test, test_predictions, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, test_predictions, zero_division=0), 4))
print("F1:", round(f1_score(y_test, test_predictions, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, test_probabilities), 4))

Random Forest Metrics
---------------------
Accuracy: 0.6787
Precision: 0.5696
Recall: 0.7294
F1: 0.6397
ROC-AUC: 0.751


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [24]:
error_analysis = test_df[
    [
        "content_id",
        "client_id",
        "trend_direction",
        "ctr",
        "content_age_days",
        "days_since_last_update",
        "impressions_90d",
        "avg_position"
    ]
].copy()

error_analysis["predicted_probability"] = test_probabilities
error_analysis["predicted_class"] = (
    test_probabilities >= 0.5
).astype(int)

error_analysis["actual_class"] = (
    test_df["is_declining"].values
)

# False positives:
# model predicts decline, but actual direction is not down
false_positives = error_analysis[
    (error_analysis["predicted_class"] == 1) &
    (error_analysis["actual_class"] == 0)
]

# False negatives:
# model misses an actual declining item
false_negatives = error_analysis[
    (error_analysis["predicted_class"] == 0) &
    (error_analysis["actual_class"] == 1)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nExample false positives:")
display(false_positives.head(10))

print("\nExample false negatives:")
display(false_negatives.head(10))

False positives: 501
False negatives: 246

Example false positives:


,content_id,client_id,trend_direction,ctr,content_age_days,days_since_last_update,impressions_90d,avg_position,predicted_probability,predicted_class,actual_class
69,content_95d488a56079,client_f74efabef1,stable,0.89,140,8,564,12.1,0.606492,1,0
147,content_bec800684271,client_f74efabef1,up,1.79,91,20,56,27.4,0.606520,1,0
168,content_d7cbd76b788d,client_f74efabef1,stable,0.11,144,20,17992,6.4,0.646036,1,0
198,content_c3e86d4031b6,client_f74efabef1,new,0.00,175,20,801,10.2,0.662896,1,0
251,content_7dff534db3ae,client_f74efabef1,up,0.31,92,20,13347,5.3,0.517010,1,0
339,content_f4e177f6d346,client_f74efabef1,new,0.18,175,20,545,29.0,0.614865,1,0
407,content_06d10cba9e03,client_d4735e3a26,up,2.94,309,20,34,10.6,0.540746,1,0
465,content_29102284b855,client_f74efabef1,new,0.29,175,20,348,21.6,0.638549,1,0
494,content_218ca439f951,client_f74efabef1,new,0.00,175,20,1619,40.4,0.610137,1,0
577,content_5920115cc1ad,client_f74efabef1,new,0.00,175,20,163,44.3,0.702724,1,0



Example false negatives:


,content_id,client_id,trend_direction,ctr,content_age_days,days_since_last_update,impressions_90d,avg_position,predicted_probability,predicted_class,actual_class
48,content_326fa2fa449f,client_98a3ab7c34,down,0.00,91,1,4,8.3,0.345963,0,1
279,content_421272323961,client_d4735e3a26,down,16.67,116,8,6,9.3,0.376311,0,1
318,content_d2e655334ee2,client_d4735e3a26,down,12.50,489,20,8,2.6,0.421007,0,1
424,content_4148b1c5a86e,client_0b918943df,down,0.00,314,20,7,11.3,0.375829,0,1
477,content_aaee0ce51abf,client_d4735e3a26,down,0.00,489,20,2,7.5,0.212814,0,1
634,content_14a42157a627,client_98a3ab7c34,down,12.50,111,1,8,4.1,0.368141,0,1
829,content_dbe45abd03e0,client_d4735e3a26,down,0.00,298,20,3,3.3,0.223639,0,1
899,content_f64f410744d9,client_d4735e3a26,down,15.38,489,20,26,5.3,0.449256,0,1
1173,content_5655ccde4244,client_d4735e3a26,down,20.00,316,20,5,4.6,0.277782,0,1
1239,content_2b5f1b817381,client_d4735e3a26,down,25.00,283,20,4,2.8,0.211711,0,1


In [25]:
# Get feature names after preprocessing

feature_names = pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

importances = pipeline.named_steps[
    "model"
].feature_importances_

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(
    "importance",
    ascending=False
)

display(feature_importance.head(15))

,feature,importance
9,numeric__days_with_impressions,0.136913
5,numeric__impressions_90d,0.123024
14,numeric__avg_position,0.119286
11,numeric__content_age_days,0.093083
30,categorical__age_tier_365+,0.038775
6,numeric__clicks_90d,0.036131
16,numeric__scroll_rate,0.035724
4,numeric__char_count,0.035454
13,numeric__ctr,0.035169
12,numeric__days_since_last_update,0.032610


##Interpretation of Feature Importance

The most important features were days with impressions, impressions over 90 days, average search position, and content age.

The model appears to use a combination of traffic exposure, search performance, and content age when identifying potentially declining content. This is useful because the model is not relying on a single signal.

However, feature importance only shows how useful a feature was to the model's predictions. It does not establish causation. For example, the high importance of impressions does not mean that increasing or decreasing impressions causes content to decline.

The results are therefore interpreted as directional evidence about which available signals the model used when ranking content.

##Conclusion

The Random Forest performed better than the Week-4 rule-based baseline on the same client-held-out test set.

The Week-4 baseline achieved Precision@20 of 0.40, Precision@50 of 0.30, and Precision@100 of 0.28. The Random Forest achieved Precision@20 of 0.75, Precision@50 of 0.76, and Precision@100 of 0.74.

This means that the Random Forest placed a much larger proportion of actually declining content near the top of the ranked queue. The improvement was especially noticeable at the top 50 and top 100 items.

The Random Forest had an accuracy of 0.6787, precision of 0.5696, recall of 0.7294, F1 score of 0.6397, and ROC-AUC of 0.751.

The model produced 501 false positives and 246 false negatives. The false positives show that some pages were predicted as declining even though their observed trend was stable, up, or new. For example, some false positives had relatively old content or high impressions, which may have caused the model to assign them a higher probability of decline.

The false negatives show that some content items were actually declining but received a probability below the classification threshold. Several examples had very low impressions, which suggests that limited performance history may make decline harder for the model to identify.

The feature importance results show that the model relied most heavily on days with impressions (0.1369), impressions over 90 days (0.1230), average position (0.1193), and content age (0.0931). These are associations used by the model and should not be interpreted as proof that these variables cause content to decline.

Overall, the Random Forest appears to provide a stronger ranking signal than my Week-4 rule-based baseline on this test set. However, it should still be treated as a decision-support tool rather than an automatic instruction to refresh content. The errors show that additional context could still change whether a page is actually a good refresh candidate.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.